# DENV lineage assignment and go/no-go check (South Asia)
Kaggle, Internet ON. Input: denv_southasia_raw.csv from the previous notebook (same /kaggle/working folder, or upload it as a dataset and set CSV_PATH).

In [ ]:
!pip install -q biopython
import os, io, re, time, subprocess, glob
import pandas as pd
from Bio import Entrez
Entrez.email = "mdabusayemedu@gmail.com"

CSV_PATH = "/kaggle/working/denv_southasia_raw.csv"
if not os.path.exists(CSV_PATH):
    cands = glob.glob("/kaggle/input/**/denv_southasia_raw.csv", recursive=True)
    assert cands, "Upload denv_southasia_raw.csv as a Kaggle dataset or set CSV_PATH"
    CSV_PATH = cands[0]
df = pd.read_csv(CSV_PATH)
use = df[(df.type != "other partial") & (df.year >= 2015)].copy()
print(len(use), "usable records since 2015")
print(use.groupby("serotype").size())


In [ ]:
# Download the sequences of the usable records as FASTA (one file per serotype)
os.makedirs("work", exist_ok=True)
for s, sub in use.groupby("serotype"):
    accs = list(dict.fromkeys(sub.acc))
    with open(f"work/seqs_denv{s}.fasta", "w") as out:
        for i in range(0, len(accs), 200):
            for attempt in range(3):
                try:
                    h = Entrez.efetch(db="nuccore", id=",".join(accs[i:i+200]), rettype="fasta", retmode="text")
                    out.write(h.read()); h.close(); break
                except Exception as e:
                    print("retry", e); time.sleep(3)
            time.sleep(0.4)
    print("DENV-%d" % s, len(accs), "sequences written")


In [ ]:
# Get the Nextclade binary and the dengue datasets (names confirmed on clades.nextstrain.org)
!curl -sL https://github.com/nextstrain/nextclade/releases/latest/download/nextclade-x86_64-unknown-linux-gnu -o nextclade
!chmod +x nextclade
for s in [1, 2, 3, 4]:
    r = subprocess.run(["./nextclade", "dataset", "get", "--name", f"community/v-gen-lab/dengue/denv{s}", "--output-dir", f"work/ds{s}"], capture_output=True, text=True)
    print("DENV-%d dataset:" % s, r.returncode, (r.stderr or r.stdout)[-200:])


In [ ]:
# Run Nextclade per serotype
for s in [1, 2, 3, 4]:
    fa = f"work/seqs_denv{s}.fasta"
    if not os.path.exists(fa):
        continue
    r = subprocess.run(["./nextclade", "run", "-D", f"work/ds{s}", "--output-tsv", f"work/out{s}.tsv", fa], capture_output=True, text=True)
    print("DENV-%d run:" % s, r.returncode, (r.stderr or "")[-200:])
res = []
for s in [1, 2, 3, 4]:
    p = f"work/out{s}.tsv"
    if os.path.exists(p):
        t = pd.read_csv(p, sep="\t"); t["serotype"] = s; res.append(t)
nc = pd.concat(res, ignore_index=True)
print(list(nc.columns)[:40])


In [ ]:
# Pick the lineage column automatically and merge with country and year
lin_cols = [c for c in nc.columns if "lineage" in c.lower()] or [c for c in nc.columns if c.lower() in ("clade", "subclade")]
print("Candidate lineage columns:", lin_cols)
LIN = lin_cols[0]
nc["acc"] = nc["seqName"].astype(str).str.split().str[0]
m = use.merge(nc[["acc", LIN]], on="acc", how="inner").rename(columns={LIN: "lineage"})
m["window"] = (m.year // 3 * 3).astype(int)
print("Assigned:", len(m), "of", len(use))
print(m.lineage.value_counts().head(25))


In [ ]:
# Go/no-go rule: at least 3 countries, each with at least 2 lineages having 20+ sequences, across at least 3 time windows
def check(d):
    ok = []
    for c, g in d.groupby("country"):
        big = g.groupby("lineage").size()
        big = big[big >= 20].index
        wins = g[g.lineage.isin(big)].window.nunique()
        ok.append((c, len(big), wins))
    return pd.DataFrame(ok, columns=["country", "lineages_20plus", "windows"])
for s, d in m.groupby("serotype"):
    r = check(d)
    print("\nDENV-%d" % s); print(r.to_string(index=False))
    print("PASS" if ((r.lineages_20plus >= 2) & (r.windows >= 3)).sum() >= 3 else "FAIL", "(rule: 3 countries with 2 lineages and 3 windows)")
print(pd.crosstab([m.serotype, m.lineage], m.country).to_string())
m.to_csv("lineage_assigned.csv", index=False)
